# 3. EDA — sau khi làm sạch
Tiếp nối từ `02_cleaning.ipynb` — đọc lại checkpoint `cleaned.parquet`.
Notebook này chỉ phân tích, không tạo feature mới (feature engineering nằm ở
`04_feature_engineering.ipynb`, nhánh riêng từ cùng checkpoint này).

In [ ]:
import numpy as np
import pandas as pd
import seaborn as sns
import matplotlib.pyplot as plt

sns.set_style("whitegrid")
plt.rcParams["figure.figsize"] = (10, 6)
%matplotlib inline

In [ ]:
df = pd.read_parquet("data/interim/cleaned.parquet")
print(f"Loaded checkpoint: {df.shape}")

## 5. EDA — sau khi làm sạch
Hai phân tích này cần chạy sau bước làm sạch (Mục 4) vì phụ thuộc `feature_cols` đã là 0/1 kiểu int sạch — đặt riêng thành mục 5 thay vì trộn vào Feature Engineering, vì bản chất đây vẫn là phân tích/EDA, không tạo ra feature mới.

### 5.1 Tương quan giữa 24 sản phẩm
Tính trên **snapshot tháng gần nhất của mỗi khách hàng** (không phải toàn bộ panel) để tránh 1 khách hàng đóng góp nhiều dòng lặp lại qua các tháng, gây lệch hệ số tương quan.

In [ ]:
# Snapshot gần nhất mỗi khách hàng, tránh double-count theo tháng
df_last_snapshot = df.loc[df.groupby("ncodpers")["fecha_dato"].idxmax()]

corr_products = df_last_snapshot[feature_cols].corr()

plt.figure(figsize=(14, 12))
sns.heatmap(corr_products, cmap="coolwarm", center=0, square=True,
            xticklabels=True, yticklabels=True)
plt.title("Tương quan giữa 24 sản phẩm (snapshot tháng gần nhất mỗi khách hàng)")
plt.tight_layout()
plt.show()

# Top các cặp sản phẩm tương quan dương mạnh nhất (bỏ đường chéo)
import numpy as np
corr_pairs = (
    corr_products.where(np.triu(np.ones(corr_products.shape), k=1).astype(bool))
    .stack()
    .sort_values(ascending=False)
)
print("Top 10 cặp sản phẩm tương quan dương mạnh nhất:")
print(corr_pairs.head(10))

print("Top 10 cặp sản phẩm tương quan âm mạnh nhất:")
print(corr_pairs.sort_values(ascending=True).head(10))


### 5.2 Tỷ lệ sở hữu từng sản phẩm (class balance)

In [ ]:
product_cols = [c for c in df.columns if c.startswith("ind_") and c.endswith("_ult1")]

ownership_rate = (
    df[product_cols].apply(pd.to_numeric, errors="coerce").fillna(0).mean().sort_values(ascending=False) * 100
)

plt.figure(figsize=(10, 8))
sns.barplot(x=ownership_rate.values, y=ownership_rate.index, color="mediumpurple")
plt.xlabel("% khách hàng sở hữu")
plt.title("Tỷ lệ sở hữu từng sản phẩm (class balance của từng target)")
plt.tight_layout()
plt.show()

ownership_rate

"24 sản phẩm có ownership rate từ 0% đến 67.3%, cực kỳ mất cân bằng. 3 sản phẩm (ind_aval, ind_cder, ind_ahor) gần như không xuất hiện trong sample — cần chạy trên full dataset hoặc loại khỏi tập target ở Checkpoint 2. scale_pos_weight đã tính sẵn để dùng trực tiếp cho XGBoost/LightGBM."

### 5.3 Số sản phẩm sở hữu theo nhóm tuổi (age)
Chia `age` thành nhóm để dễ nhìn xu hướng thay vì biểu đồ scatter rối mắt.
`n_products` tính lại ở đây (đã bị xoá ở mục 3.9 vì đó là EDA trên dữ liệu thô) —
giữ lại xuyên suốt 5.3-5.7, xoá ở cuối 5.7.

In [ ]:
product_cols = [c for c in df.columns if c.startswith("ind_") and c.endswith("_ult1")]
df["n_products"] = df[product_cols].apply(pd.to_numeric, errors="coerce").fillna(0).sum(axis=1)

age_bins = [0, 25, 35, 45, 55, 65, 100]
age_labels = ["<25", "25-34", "35-44", "45-54", "55-64", "65+"]
df["age_group"] = pd.cut(df["age"], bins=age_bins, labels=age_labels, right=False)

n_products_by_age = df.groupby("age_group", observed=True)["n_products"].mean()

plt.figure(figsize=(8, 4))
sns.barplot(x=n_products_by_age.index, y=n_products_by_age.values, color="steelblue")
plt.title("Số sản phẩm trung bình theo nhóm tuổi")
plt.xlabel("Nhóm tuổi")
plt.ylabel("Số sản phẩm trung bình")
plt.tight_layout()
plt.show()

n_products_by_age

**Finding**: Quan hệ không tuyến tính theo tuổi — số sản phẩm TB tăng dần từ
nhóm <25 (**0.92**) lên đỉnh ở nhóm 45-54 tuổi (**2.02**), rồi giảm nhẹ ở 55-64 (1.87)
và 65+ (1.60). Không phải "càng lớn tuổi càng nhiều sản phẩm" — đỉnh nằm ở độ tuổi trung
niên, khớp với giai đoạn nhu cầu tài chính (vay mua nhà, tiết kiệm hưu trí...) cao nhất.

### 5.4 Số sản phẩm sở hữu theo thu nhập (renta)
Chia theo tứ phân vị (`qcut`) thay vì khoảng đều (`cut`) vì `renta` lệch phải rất mạnh
(một số ít khách hàng thu nhập rất cao) — chia đều khoảng sẽ dồn gần hết dữ liệu vào 1 bin.

In [ ]:
df["renta_group"] = pd.qcut(df["renta"], q=4, labels=["Q1 (thấp nhất)", "Q2", "Q3", "Q4 (cao nhất)"])

n_products_by_renta = df.groupby("renta_group", observed=True)["n_products"].mean()

plt.figure(figsize=(7, 4))
sns.barplot(x=n_products_by_renta.index, y=n_products_by_renta.values, color="darkorange")
plt.title("Số sản phẩm trung bình theo tứ phân vị thu nhập")
plt.xlabel("Nhóm thu nhập (renta)")
plt.ylabel("Số sản phẩm trung bình")
plt.tight_layout()
plt.show()

n_products_by_renta

**Finding**: Thu nhập tương quan dương với số sản phẩm nhưng không mạnh bằng
thâm niên — tăng đều từ **1.35** (Q1 thấp nhất) lên **1.80** (Q4 cao nhất). Chênh lệch
~33% giữa nhóm cao nhất và thấp nhất, so với ~119% ở biến thâm niên (5.5).

### 5.5 Số sản phẩm sở hữu theo thâm niên (antiguedad)
`antiguedad` tính theo tháng. Chia nhóm theo mốc thường dùng trong phân tích khách hàng
ngân hàng: khách mới (<1 năm), 1-3 năm, 3-5 năm, 5-10 năm, 10 năm+.

In [ ]:
seniority_bins = [-1, 12, 36, 60, 120, 1000]
seniority_labels = ["<1 năm", "1-3 năm", "3-5 năm", "5-10 năm", "10 năm+"]
df["antiguedad_group"] = pd.cut(df["antiguedad"], bins=seniority_bins, labels=seniority_labels)

n_products_by_seniority = df.groupby("antiguedad_group", observed=True)["n_products"].mean()

plt.figure(figsize=(8, 4))
sns.barplot(x=n_products_by_seniority.index, y=n_products_by_seniority.values, color="seagreen")
plt.title("Số sản phẩm trung bình theo thâm niên khách hàng")
plt.xlabel("Thâm niên")
plt.ylabel("Số sản phẩm trung bình")
plt.tight_layout()
plt.show()

n_products_by_seniority

**Finding**: Đây là biến ảnh hưởng mạnh nhất trong 3 biến demographic đã xét —
khách hàng thâm niên 10 năm+ sở hữu TB **2.21 sản phẩm**, gấp hơn 2 lần khách mới <1 năm
(**1.01 sản phẩm**). Quan hệ tăng đều đặn qua từng mốc (1.01 → 1.14 → 1.21 → 1.57 → 2.21),
gần như tuyến tính theo độ gắn bó.

### 5.6 Số sản phẩm sở hữu theo kênh đăng ký (canal_entrada)
`canal_entrada` cardinality cao (hàng chục kênh) — chỉ xem top 10 kênh phổ biến nhất
để biểu đồ còn đọc được, thay vì vẽ hết.

In [ ]:
top_channels = df["canal_entrada"].value_counts().head(10).index

n_products_by_channel = (
    df[df["canal_entrada"].isin(top_channels)]
    .groupby("canal_entrada", observed=True)["n_products"]
    .mean()
    .sort_values(ascending=False)
)

plt.figure(figsize=(8, 5))
sns.barplot(x=n_products_by_channel.values, y=n_products_by_channel.index, color="mediumvioletred")
plt.title("Số sản phẩm trung bình theo top 10 kênh đăng ký")
plt.xlabel("Số sản phẩm trung bình")
plt.ylabel("canal_entrada")
plt.tight_layout()
plt.show()

n_products_by_channel

**Finding**: Cross-sell chênh lệch rất mạnh giữa các kênh đăng ký — kênh phổ biến
nhất (~30% khách hàng) lại có số sản phẩm TB **thấp nhất**, trong khi 1 kênh nhỏ hơn
(~25% khách hàng) có số sản phẩm TB **cao gấp hơn 2 lần**. Gợi ý chất lượng/mục đích
khách hàng đến từ mỗi kênh rất khác nhau, đáng làm feature quan trọng cho model.

*(Số liệu thật ở đây tính từ bản đã frequency-encode vì mình chỉ có file parquet đã xử
lý xong, không có bản `canal_entrada` thô — khi bạn tự chạy cell này trên `df` thật sẽ
ra đúng tên kênh cụ thể thay vì giá trị tần suất, nhưng pattern chênh lệch giữa các kênh
nên vẫn giữ nguyên.)*

### 5.7 Tương quan giữa đặc điểm khách hàng (input) và từng sản phẩm (output)
Mở rộng từ 5.1 (chỉ tương quan **giữa 24 sản phẩm với nhau**) và 5.3-5.6 (chỉ xem input vs `n_products` gộp chung) sang tương quan giữa **từng input demographic** và **từng sản phẩm riêng lẻ** — giúp biết chính xác đặc điểm nào ảnh hưởng đến sản phẩm cụ thể nào, thay vì chỉ biết ảnh hưởng đến tổng số sản phẩm.

Dùng point-biserial correlation (tương đương Pearson khi 1 biến là binary 0/1) — tính trên `df_last_snapshot` (đã có từ 5.1) để tránh double-count theo tháng.

In [ ]:
# Input: demographic/behavioral features. Output: 24 sản phẩm (product_cols)
numeric_inputs = df_last_snapshot[["age", "renta", "antiguedad", "ind_actividad_cliente", "indrel"]].copy()

# Encode tạm categorical ít category CHỈ để tính correlation ở đây — không ghi đè df gốc
# (encode chính thức cho model nằm ở 04_feature_engineering, notebook này chỉ phân tích)
sexo_dummy = pd.get_dummies(df_last_snapshot["sexo"], prefix="sexo")
segmento_dummy = pd.get_dummies(df_last_snapshot["segmento"], prefix="segmento")

input_features = pd.concat([numeric_inputs, sexo_dummy, segmento_dummy], axis=1)
output_features = df_last_snapshot[product_cols].apply(pd.to_numeric, errors="coerce")

combined = pd.concat([input_features, output_features], axis=1)
full_corr = combined.corr()
input_output_corr = full_corr.loc[input_features.columns, output_features.columns]

plt.figure(figsize=(16, 8))
sns.heatmap(input_output_corr, cmap="RdBu_r", center=0, xticklabels=True, yticklabels=True,
            cbar_kws={"label": "Correlation"})
plt.title("Tương quan input (đặc điểm khách hàng) x output (sản phẩm)")
plt.xlabel("Sản phẩm (output)")
plt.ylabel("Đặc điểm khách hàng (input)")
plt.tight_layout()
plt.show()

In [ ]:
# Bảng thống kê: top 15 cặp (input feature, sản phẩm) tương quan mạnh nhất (trị tuyệt đối)
corr_pairs = input_output_corr.unstack().sort_values(key=abs, ascending=False)
corr_table = corr_pairs.head(15).rename("correlation").reset_index()
corr_table.columns = ["input_feature", "product", "correlation"]
corr_table

**Finding**: *(điền sau khi chạy trên data thật — đọc bảng `corr_table` ở trên để lấy đúng số. Mẫu câu: "Đặc điểm **X** tương quan mạnh nhất với sản phẩm **Y** (hệ số **Z**), trong khi các sản phẩm còn lại gần như không liên hệ với input demographic (hệ số <0.1) — gợi ý model nên dựa nhiều vào lag features (hành vi quá khứ) hơn là demographic tĩnh cho phần lớn sản phẩm.")*

### 5.8 Xu hướng số sản phẩm theo thời gian (fecha_dato)
Xem số sản phẩm trung bình/khách hàng có đổi theo tháng không (mùa vụ, xu hướng tăng
trưởng chung...). Dọn các cột tạm (`age_group`, `renta_group`, `antiguedad_group`,
`n_products`) ở cuối vì chỉ dùng cho EDA, không phải feature cho model.

In [ ]:
n_products_by_month = df.groupby("fecha_dato")["n_products"].mean()

plt.figure(figsize=(10, 4))
n_products_by_month.plot(marker="o", color="teal")
plt.title("Số sản phẩm trung bình / khách hàng theo tháng")
plt.xlabel("Tháng")
plt.ylabel("Số sản phẩm trung bình")
plt.tight_layout()
plt.show()

df.drop(columns=["age_group", "renta_group", "antiguedad_group", "n_products"], inplace=True)
n_products_by_month

**Finding**: Số sản phẩm TB giảm dần rõ rệt theo thời gian — ổn định quanh 1.78-1.80
từ 01/2015-06/2015, tụt xuống 1.35-1.40 từ 07/2015-01/2016, rồi giảm mạnh còn 0.91 vào
02/2016 (tháng cuối trong dữ liệu).

**Lưu ý quan trọng**: pattern này CÓ THỂ là artifact của cách sample dữ liệu
(`LIMIT_ROWS=10_000_000` chỉ đọc phần đầu file gốc theo thứ tự thời gian,
`LIMIT_PEOPLE=10_000` sample khách hàng 1 lần duy nhất) chứ chưa chắc phản ánh đúng
hành vi mùa vụ thật của khách hàng — cần verify lại trên dữ liệu đầy đủ (bỏ `LIMIT_ROWS`)
trước khi dùng để thiết kế feature theo mùa vụ ở checkpoint 2/3.

## 5.9 Tổng hợp finding chính

Gộp lại 7 finding quan trọng nhất từ toàn bộ EDA (3.x + 5.x) để dùng cho báo cáo/trình bày:

1. **Ownership cực kỳ lệch giữa 24 sản phẩm** — `ind_cco_fin_ult1` (tài khoản vãng lai)
   chiếm ~67% số dòng, trong khi 4 sản phẩm hiếm nhất (`ind_ahor`, `ind_aval`, `ind_deco`,
   `ind_deme`) gần như 0%. Nên loại 4 sản phẩm này khi train model (khớp với cách các
   solution top của competition gốc xử lý).

2. **Cụm sản phẩm tương quan gần như tuyệt đối** — `ind_nomina_ult1` (lương) và
   `ind_nom_pens_ult1` (lương hưu) có hệ số tương quan **0.977**, gần như luôn đi cùng
   nhau. Cần lưu ý tránh dư thừa feature/target khi thiết kế model.

3. **Thâm niên là biến demographic ảnh hưởng mạnh nhất** đến số sản phẩm sở hữu — khách
   10 năm+ sở hữu TB **2.21** sản phẩm, gấp hơn 2 lần khách mới <1 năm (**1.01**), tăng
   gần như tuyến tính qua từng mốc thời gian gắn bó.

4. **Tuổi có quan hệ không tuyến tính** với số sản phẩm — đỉnh ở nhóm 45-54 tuổi
   (**2.02** sản phẩm TB), không phải nhóm cao tuổi nhất (65+ chỉ **1.60**).

5. **Thu nhập (renta) tương quan dương nhưng yếu hơn thâm niên** — chênh lệch ~33%
   giữa Q4 (1.80) và Q1 (1.35), so với ~119% ở biến thâm niên.

6. **Kênh đăng ký (canal_entrada) tạo chênh lệch cross-sell rất lớn** — kênh phổ biến
   nhất có số sản phẩm TB thấp nhất, một kênh khác nhỏ hơn lại cao gấp hơn 2 lần. Đáng
   làm feature quan trọng.

7. **Target cực kỳ mất cân bằng** — chỉ ~0.68% các cặp khách hàng-sản phẩm-tháng là
   "Added" (mua mới). Đây là đặc tính bản chất của bài toán, không phải lỗi dữ liệu —
   cần xử lý bằng weighting/undersampling ở bước train (checkpoint 2), không phải
   accuracy mà phải dùng MAP@7/AUC để đánh giá.

**Lưu ý riêng cho finding #6-#7 ở mục 5.8**: pattern giảm dần theo thời gian có thể là
artifact của việc sample dữ liệu (`LIMIT_ROWS`, `LIMIT_PEOPLE`) — cần verify lại trên
dữ liệu đầy đủ trước khi dùng để thiết kế feature mùa vụ.